# EDA general — comparación de los 10 datasets

Este notebook no analiza un dataset en detalle, sino que los **compara**:
1. Estado de descarga (desde `manifest.json`).
2. Tamaño, número de filas y columnas de los archivos principales de cada dataset.
3. Consolidación de los `tables/resumen.csv` que genera cada notebook individual (ejecútalos primero).

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context(None)   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Manifiesto de descargas

In [ ]:
manifest = read_json_any(ctx.data_dir / "manifest.json")
man = pd.DataFrame.from_dict(manifest["datasets"], orient="index").sort_values("rank")
display(man[["rank", "category", "size_mb", "file_count", "downloaded_at"]])

## 2. Archivos principales de cada dataset
Contar líneas de los archivos grandes toma unos segundos.

In [ ]:
MAIN_FILES = {
    "01": [
        "default/jailbreak_dataset_full.csv"
    ],
    "02": [
        "embeddings.parquet"
    ],
    "03": [
        "secure_programming_dpo.json"
    ],
    "04": [
        "all_cve_database.jsonl"
    ],
    "05": [
        "220426-CyberSec-Dataset_escaped.jsonl"
    ],
    "06": [
        "CyberSec-Dataset_escaped.jsonl"
    ],
    "07": [
        "massive_training.jsonl",
        "millions_final.jsonl",
        "millions_training.jsonl",
        "training_data.jsonl"
    ],
    "08": [],
    "09": [
        "advanced_siem_dataset.jsonl"
    ],
    "10": [
        "train.jsonl",
        "valid.jsonl"
    ]
}

rows = []
for prefix, rels in MAIN_FILES.items():
    d = dataset_dir(prefix, ctx.root)
    all_files = list_files(d)
    if not rels:  # ClawTrojan: estructura de carpetas, se cuentan trayectorias
        n = len(list((d / "samples").glob("*/*.json")))
        rows.append({"#": prefix, "dataset": d.name, "archivo": "samples/*/*.json", "size_mb": all_files.size_mb.sum(),
                     "filas": n, "columnas": "(estructura de carpetas)"})
        continue
    for rel in rels:
        p = d / rel
        if not p.exists():
            continue
        if p.suffix == ".parquet":
            t = pd.read_parquet(p); n, cols = len(t), list(t.columns)
        elif p.suffix == ".csv":
            t = pd.read_csv(p); n, cols = len(t), list(t.columns)
        else:
            n, cols = count_lines(p), list(peek_record(p).keys())
        rows.append({"#": prefix, "dataset": d.name, "archivo": rel, "size_mb": round(p.stat().st_size / 1e6, 1),
                     "filas": n, "columnas": ", ".join(cols)})
overview = pd.DataFrame(rows)
display(overview)
save_table(overview, ctx, "archivos_principales", index=False)

In [ ]:
agg = overview.groupby("dataset").agg(size_mb=("size_mb", "sum"), filas=("filas", "sum"))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
agg["size_mb"].sort_values().plot.barh(ax=axes[0], logx=True); axes[0].set_title("Tamaño de archivos principales (MB, log)")
agg["filas"].sort_values().plot.barh(ax=axes[1], logx=True); axes[1].set_title("Filas / registros (log)")
plt.tight_layout(); save_fig(fig, ctx, "tamanio_y_filas"); plt.show()

## 3. Consolidar los resúmenes de cada notebook

In [ ]:
res = []
for p in sorted((ctx.root / "eda").glob("[0-9][0-9]_*/tables/resumen.csv")):
    if p.parent.parent.name.startswith("00"):
        continue
    r = pd.read_csv(p); r.insert(0, "carpeta", p.parent.parent.name); res.append(r)
if res:
    resumen = pd.concat(res, ignore_index=True)
    display(resumen)
    save_table(resumen, ctx, "resumen_consolidado", index=False)
else:
    print("Aún no hay resúmenes: ejecuta primero los notebooks de cada dataset.")

## 4. Conclusiones generales
- 